# 05 — Trade Archetypes: Bounce & Breakout

**Strategy reference:** §11.1 (Bounce), §11.2 (Breakout),
§9.6.3 (Hold/Break score).

Ripple has exactly two trade archetypes in V1:

| Archetype | When | Direction |
|---|---|---|
| **Bounce**   | wall holds and absorbs flow | *away* from wall |
| **Breakout** | wall fails (pulled or consumed) | *through* wall |

Both follow the same SETUP → CONFIRMATION → ENTRY pipeline.
Pseudocode here mirrors §11.1.1 and §11.2.1.

In [ ]:
# ── Data-source configuration ─────────────────────────────────────────
# OHLCV (Parquet) — S3 or local, controlled by DATA_STORE env var:
#   Local (default):  reads <project_root>/data/ohlcv/...
#   S3:               uncomment the two lines below
# import os
# os.environ["DATA_STORE"] = "s3"
# os.environ["S3_BUCKET"]  = "trading-data-centheos"
#
# Tick data (HDF5) — always stored locally; pull from S3 on demand:
#   load_ticks(...)              → use local cache (fast, no network)
#   load_ticks(..., refresh=True) → sync from S3 then read (ETag-gated)
#   Requires: AWS_PROFILE=trading (or AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY)
import os
os.environ["AWS_PROFILE"] = "trading"
os.environ["S3_BUCKET"]   = "trading-data-centheos"
# ─────────────────────────────────────────────────────────────────────

import sys, importlib
from pathlib import Path

_here = Path.cwd().resolve()
for _cand in [_here, *_here.parents]:
    if (_cand / "schemas.py").exists():
        _root = _cand; break
else:
    raise RuntimeError("Could not locate project root (no schemas.py found)")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import notebooks.utils as _utils_mod
importlib.reload(_utils_mod)   # always pick up on-disk changes without restarting the kernel

from notebooks.utils import (
    load_ohlcv, list_ohlcv, load_ticks, latest_book,
    plot_ohlcv, plot_equity_curve, configure_pandas, env_summary,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

configure_pandas()
%matplotlib inline

In [ ]:
from schemas import RippleConfig, TradeArchetype, TradeSide
cfg = RippleConfig()
print('proximity_sigma =', cfg.proximity_sigma)
print('bounce_microprice_shift_ticks =', cfg.bounce_microprice_shift_ticks)
print('breakout_depth_fail_ratio =', cfg.breakout_depth_fail_ratio)
print('breakout_cvd_slope_thresh =', cfg.breakout_cvd_slope_thresh)

## 1. Bounce detection (§11.1.1)
```
if wall.quality < wall_min_quality:    skip
if |microprice - wall.price| > proximity_sigma * σ_P:   skip
if liquidity_state not in {ABSORPTION, STABLE}:    skip
→ create BounceSetup
```

Confirmation requires three things simultaneously:
1. flow declining (CVD slope opposite to side)
2. microprice shifted away from wall by ≥ `bounce_microprice_shift_ticks`
3. imbalance favouring the bounce direction

In [ ]:
from dataclasses import dataclass

@dataclass
class BounceSetup:
    side: str           # 'LONG' or 'SHORT'
    wall_price: float
    setup_ts: int
    side_sign: int

def detect_bounce(walls, microprice, sigma_P, liquidity_state, cfg):
    for w in walls:
        if w['quality'] < cfg.wall_min_quality:
            continue
        if abs(microprice - w['price']) > cfg.proximity_sigma * sigma_P:
            continue
        if liquidity_state not in {'ABSORPTION', 'STABLE'}:
            continue
        side = 'LONG' if w['side'] == 'BID' else 'SHORT'
        return BounceSetup(side, w['price'], 0, +1 if side == 'LONG' else -1)
    return None

def confirm_bounce(setup, microprice, cvd_slope, imbalance, tick_size, cfg):
    flow_declining = (cvd_slope * setup.side_sign < 0
                       or abs(cvd_slope) < cfg.exhaustion_cvd_slope_thresh)
    shift_units = (microprice - setup.wall_price) * setup.side_sign
    shifted = shift_units > cfg.bounce_microprice_shift_ticks * tick_size
    imb_ok  = imbalance * setup.side_sign > cfg.bounce_imbalance_thresh
    return bool(flow_declining and shifted and imb_ok)

walls_demo = [
    {'price': 100.0, 'side': 'BID', 'quality': 0.9},
    {'price': 102.0, 'side': 'ASK', 'quality': 0.4},  # below quality threshold
]
setup = detect_bounce(walls_demo, microprice=100.4, sigma_P=1.0,
                       liquidity_state='ABSORPTION', cfg=cfg)
print('detected setup:', setup)
print('confirmed?    :', confirm_bounce(setup, microprice=100.6,
                                        cvd_slope=-0.02, imbalance=+0.25,
                                        tick_size=0.01, cfg=cfg))

## 2. Breakout detection (§11.2.1)
Mirror logic: the wall must be *failing* (depth ratio below
`breakout_depth_fail_ratio` or below the absolute floor), price
must be at or through the wall, then confirm with CVD acceleration
in the breakout direction.

In [ ]:
@dataclass
class BreakoutSetup:
    side: str
    wall_price: float
    initial_depth: float
    side_sign: int
    setup_ts: int = 0

def detect_breakout(walls, microprice, tick_size, cfg):
    for w in walls:
        if w['quality'] < cfg.wall_min_quality:
            continue
        if w['depth'] / w['initial_depth'] > cfg.breakout_depth_fail_ratio:
            continue
        if w['side'] == 'ASK' and microprice < w['price'] - tick_size:
            continue
        if w['side'] == 'BID' and microprice > w['price'] + tick_size:
            continue
        side = 'LONG' if w['side'] == 'ASK' else 'SHORT'
        return BreakoutSetup(side, w['price'], w['initial_depth'], +1 if side == 'LONG' else -1)
    return None

def confirm_breakout(setup, last_trade_price, current_depth, cvd_slope, cfg):
    through = (last_trade_price - setup.wall_price) * setup.side_sign > 0
    wall_failed = (current_depth < cfg.breakout_depth_fail_abs
                    or current_depth / setup.initial_depth < cfg.breakout_depth_fail_ratio)
    flow_accel = cvd_slope * setup.side_sign > cfg.breakout_cvd_slope_thresh
    return bool(through and wall_failed and flow_accel)

walls_demo = [
    {'price': 105.0, 'side': 'ASK', 'depth': 1.0, 'initial_depth': 10.0, 'quality': 0.8},
]
setup = detect_breakout(walls_demo, microprice=105.0, tick_size=0.01, cfg=cfg)
print('setup:', setup)
print('confirmed?:', confirm_breakout(setup, last_trade_price=105.5,
                                       current_depth=1.0, cvd_slope=+0.08, cfg=cfg))

## 3. Hold-vs-Break logistic (§9.6.3)
$$P(\text{hold} \mid x) = \sigma(\beta_0 + \beta_1 W_p + \beta_2 F_t + \beta_3 R_p + \beta_4 I_t)$$

V1 uses deterministic thresholds; the logistic is a future
calibration target.  Sensitivity sweep below.

In [ ]:
def p_hold(W, F, R, I, beta=(0.0, 1.5, -1.2, 0.8, 0.4)):
    z = beta[0] + beta[1]*W + beta[2]*F + beta[3]*R + beta[4]*I
    return 1.0 / (1.0 + np.exp(-z))

F_grid = np.linspace(0, 1, 80)
for W in [0.4, 0.8, 1.2, 1.6]:
    plt.plot(F_grid, [p_hold(W, f, 0.5, 0.0) for f in F_grid], label=f'W={W}')
plt.title('P(hold) vs aggressive flow F  for varying wall quality W')
plt.xlabel('aggressive flow F'); plt.ylabel('P(hold)'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 4. Label bounce setups on real tick data
Walk the recent tick stream, look for moments where mid-price
approached a synthetic 'wall' (top-of-book bid heavier than
median), and annotate the chart.

In [ ]:
ticks = # refresh=False (default) — use local cache, no network access.
# refresh=True            — ETag-check S3 and download only if collector
#                           has uploaded new data since last refresh.
load_ticks('BTCUSDT',
                   max_trades=5_000,
                   max_depth_snapshots=20_000,
                   max_depth_updates=20_000,
                   refresh=False)
snaps  = ticks['depth_snapshots']
trades = ticks['trades']

def best_bids(snap_df):
    bids = snap_df[snap_df['side']==0]
    return (bids.groupby('timestamp')
                 .apply(lambda g: g.nlargest(1, 'price').iloc[0], include_groups=False)
                 [['price','quantity']])

bb = best_bids(snaps).rename(columns={'price':'bid_p','quantity':'bid_q'})
bb['med_q'] = bb['bid_q'].rolling(200, min_periods=20).median()
bb['ratio'] = bb['bid_q'] / bb['med_q']
candidates = bb[bb['ratio'] > cfg.wall_depth_multiple].head(20)
candidates

In [ ]:
ts_dt   = pd.to_datetime(bb.index, unit='ms')
cand_dt = pd.to_datetime(candidates.index, unit='ms')
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ts_dt, bb['bid_p'], color='#1f77b4', linewidth=0.7, label='best bid')
ax.scatter(cand_dt, candidates['bid_p'], color='#d62728', s=24, zorder=5, label='bid-wall candidates')
ax.set_title('Top-of-book best bid with detected bid walls'); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 5. Setup-frequency statistics
How often do we even *see* a wall candidate?  How often does it
persist for more than one snapshot?

In [ ]:
freq = bb['ratio'].dropna()
print(f'snapshots  total       : {len(freq):,}')
print(f'wall-ratio >= 3        : {(freq >= 3).sum():,} ({(freq >= 3).mean():.2%})')
print(f'wall-ratio >= 5        : {(freq >= 5).sum():,} ({(freq >= 5).mean():.2%})')
print(f'wall-ratio >= 10       : {(freq >= 10).sum():,} ({(freq >= 10).mean():.2%})')
freq.describe(percentiles=[0.5, 0.9, 0.99])

## Takeaways

* SETUP detection is cheap; the **confirmation** filter is what
  cuts noise. All three confirmation conditions must hold.
* The hold-vs-break logistic is the V2 generalisation: once we
  have V1 labels (this notebook's logic) we can fit β via logistic
  regression.
* Even on a few thousand snapshots, walls above ×3 median depth
  appear a non-trivial fraction of the time — so the bottleneck
  for trade frequency is *confirmation*, not detection.